#### setting tracking uri and connecting to dagshub

In [1]:
import mlflow
import dagshub
import pandas as pd
import numpy as np
import json

mlflow.set_tracking_uri("https://dagshub.com/gbera23-dev/Machine-Learning.mlflow")
dagshub.init(repo_owner='gbera23-dev', repo_name='Machine-Learning', mlflow=True)

/home/giga/FREEUNI/semester_6/ML/Machine-Learning/Assignment1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accessing as gbera23-dev

Initialized MLflow to track repo "gbera23-dev/Machine-Learning"

Repository gbera23-dev/Machine-Learning initialized!

#### loading optimized model from registry

In [2]:
model = mlflow.sklearn.load_model("models:/house_prices_linear_regression_optimized/3")

#### loading test data

In [3]:
test_df = pd.read_csv("house-prices-advanced-regression-techniques/test.csv")

# save Id before any preprocessing
test_ids = test_df["Id"]

#### preprocessing test data

##### dropping low data columns

In [4]:
outsider_col_names = ['Id']

average_nonNull_entries = (test_df.count().sum() / test_df.count().size).__round__()

[outsider_col_names.append(i) for i in test_df.columns if test_df[i].count() <= average_nonNull_entries * (40 / 100)]

for col_name in outsider_col_names:
    if test_df.columns.str.contains(col_name).any():
        test_df = test_df.drop(col_name, axis=1, inplace=False)

##### handling null values in numerical data

In [5]:
numerical_part_df = test_df.select_dtypes(include=['int64', 'float64'])
num_rows = len(test_df)

numeric_columns_with_nan = [col for col in numerical_part_df.columns if test_df[col].count() != num_rows]

for col in numeric_columns_with_nan:
    test_df[col] = test_df[col].fillna(test_df[col].median())

##### handling null values in categorical data

In [6]:
categorical_part_df = test_df.select_dtypes(exclude=['int64', 'float64'])

categorical_columns_with_nan = [col for col in categorical_part_df.columns if test_df[col].count() != num_rows]

for col in categorical_columns_with_nan:
    test_df[col] = test_df[col].fillna("None")

##### feature engineering

In [7]:
test_df['TotalSF'] = test_df['TotalBsmtSF'] + test_df['1stFlrSF'] + test_df['2ndFlrSF']
test_df['HouseAge'] = test_df['YrSold'] - test_df['YearBuilt']
test_df['RemodAge'] = test_df['YrSold'] - test_df['YearRemodAdd']
test_df['TotalBath'] = test_df['FullBath'] + 0.5 * test_df['HalfBath'] + test_df['BsmtFullBath'] + 0.5 * test_df['BsmtHalfBath']
test_df['IsRemodeled'] = (test_df['YearRemodAdd'] != test_df['YearBuilt']).astype(int)

##### one hot encoding

In [8]:
test_df = pd.get_dummies(test_df, dtype=int, drop_first=True)

##### aligning columns with training data

In [9]:
with open("linear_regression_optimized_model/train_columns_optimized.json") as f:
    train_columns = json.load(f)

test_df = test_df.reindex(columns=train_columns, fill_value=0)

# fix any remaining nulls introduced by reindex
cols_to_fill = test_df.isnull().sum()[test_df.isnull().sum() > 0].index
for col in cols_to_fill:
    test_df[col] = test_df[col].fillna(test_df[col].median())

#### predict and build submission

In [10]:
predictions = model.predict(test_df)

submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": predictions
})

submission.to_csv("submission_optimized.csv", index=False)